In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import zipfile
import os

zip_path = "/kaggle/input/my-erisk19/eRisk2019_T1.ZIP"   # replace this according to need
extract_to = "/kaggle/working/my-eRisk19_extracted"        # can be any folder



# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# DEMO CELL
### DEMO CELL

In [2]:
#Function1: extract ZIP
def extract_zip(zip_path, extract_to="/kaggle/working/extracted"):
    """
    Extracts a ZIP file to a given directory.
    """
    os.makedirs(extract_to, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_to)
    return extract_to


In [3]:
#Function1b: extract ZIP recursively - I am using this
def extract_zip_recursive(zip_path, extract_to="/kaggle/working/extracted"):
    """
    Extracts a ZIP file and recursively extracts any nested ZIP files found inside it.
    """
    os.makedirs(extract_to, exist_ok=True)

    # --- Step 1: Extract the main zip ---
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_to)

    # --- Step 2: Recursively extract inner zip files ---
    extracted_new_zip = True

    while extracted_new_zip:
        extracted_new_zip = False

        # Walk through the extracted directory
        for root, dirs, files in os.walk(extract_to):
            for f in files:
                if f.lower().endswith(".zip"):
                    zip_file_path = os.path.join(root, f)

                    # Extract into a folder named after the zip (without .zip)
                    new_folder = os.path.splitext(zip_file_path)[0]
                    os.makedirs(new_folder, exist_ok=True)

                    with zipfile.ZipFile(zip_file_path, 'r') as z:
                        z.extractall(new_folder)

                    # After extracting, delete the original zip file to avoid infinite loops
                    os.remove(zip_file_path)

                    extracted_new_zip = True  # Continue loop to check for deeper nested zips

    return extract_to

In [4]:
#Function2: collect all XML files
def collect_xml_files(folder):
    """
    Returns a list of all XML files under a directory tree.
    """
    xml_files = []
    for root, dirs, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(".xml"):
                xml_files.append(os.path.join(root, f))
    return xml_files


# BEGIN

## Read xml file in the specified path and return the dataframe
### Input-> xml file path
### Output-> dataframe containing columns ID, TITLE, DATE, INFO and TEXT

In [5]:
import xml.etree.ElementTree as ET
import pandas as pd
import glob
import os

def parse_xml(file_path,truth_dict):
    tree = ET.parse(file_path)
    root = tree.getroot()
    
    ID = root.find('ID').text
    
    # Clean ID if needed (e.g., strip spaces)
    if ID is not None:
        clean_id = ID.strip()
        label = truth_dict.get(clean_id, None)
    else:
        clean_id = None
        label = None

     
    rows = []
    for wr in reversed(root.findall('WRITING')): #In xml file records stored in reverse order
        title = wr.find('TITLE').text if wr.find('TITLE') is not None else None
        date = wr.find('DATE').text if wr.find('DATE') is not None else None
        info = wr.find('INFO').text if wr.find('INFO') is not None else None
        text = wr.find('TEXT').text if wr.find('TEXT') is not None else None
        
        rows.append([ID, title, date, info, text,label])
    
    df = pd.DataFrame(rows, columns=['ID','TITLE','DATE','INFO','TEXT','label'])
    return df




### Loop through all directory and read the xml file

In [6]:
#data_path = "/kaggle/working/my-eRisk19_extracted/test data - T1/T1_data/data/*.xml"

#all_files = glob.glob(data_path)

def form_dataframe(all_files, truth_dict):
    all_dfs = []
    
    for file in all_files:
        df = parse_xml(file,truth_dict)
        all_dfs.append(df)
    
    final_df = pd.concat(all_dfs, ignore_index=True)
    return final_df


# END

In [7]:
#Function to read the golden truth file 
def load_golden_truth(folder):
    """
    Searches for a file containing 'risk_golden_truth' or 'risk-golden-truth'
    in its name, and loads ID → label mapping as a dictionary.
    Supports both space and tab separators.
    """
    for root, dirs, files in os.walk(folder):
        for f in files:
            fname = f.lower()
            if ("risk_golden_truth" in fname) or ("risk-golden-truth" in fname):
                path = os.path.join(root, f)
                truth = {}

                with open(path, "r") as file:
                    for line in file:
                        line = line.strip()
                        if not line:
                            continue

                        # Split on tab OR space
                        parts = line.split("\t") if "\t" in line else line.split()

                        if len(parts) >= 2:
                            truth[parts[0]] = parts[1]  # id → label

                return truth

    raise FileNotFoundError("Golden truth file not found in extracted folder.")


In [8]:
extracted = extract_zip_recursive(zip_path, extract_to)
#unzip_recursive(extract_to)

In [9]:
# 1. Load golden truth labels
truth_dict_test = load_golden_truth(f"{extract_to}/test data - T1")
truth_dict_train = load_golden_truth( f"{extract_to}/training data - t1/2018 train")
truth_dict_val = load_golden_truth(f"{extract_to}/training data - t1/2018 test")

# 2. Build each split
test_xml_paths = collect_xml_files(
    f"{extract_to}/test data - T1"
)
df_test = form_dataframe(test_xml_paths, truth_dict_test)

train_xml_paths = collect_xml_files(
    f"{extract_to}/training data - t1/2018 train"
)
df_train = form_dataframe(train_xml_paths, truth_dict_train)

val_xml_paths = collect_xml_files(
    f"{extract_to}/training data - t1/2018 test"
)
df_val = form_dataframe(val_xml_paths, truth_dict_val)


In [10]:
df_train.head()

,ID,TITLE,DATE,INFO,TEXT,label
0,subject1113,,2016-04-27 22:44:47,reddit post,"I don't know really, when I started to go thr...",1
1,subject1113,,2016-04-27 22:46:36,reddit post,"Dude, I think I know myself a lot better than...",1
2,subject1113,,2016-04-27 22:48:09,reddit post,"Yeah, obviously. The world doesn't look too k...",1
3,subject1113,,2016-04-27 22:50:17,reddit post,"By supporting that law, are you saying that m...",1
4,subject1113,,2016-04-28 09:31:01,reddit post,Nah I rather not.,1


In [11]:
df_test.head()

,ID,TITLE,DATE,INFO,TEXT,label
0,subject8835,Anyone else think we are due for a Seinfeld m...,2012-02-19 06:15:53,reddit post,I for one was a huge fan and was extremely di...,0
1,subject8835,"Iaman Inupiaq (eskimo) Born in Barrow, AK liv...",2012-02-24 17:15:37,reddit post,[removed],0
2,subject8835,Kona Brewfest 2012,2012-03-11 21:37:53,reddit post,,0
3,subject8835,Sand castle with working volcano and Beer tap.,2012-03-12 00:02:52,reddit post,,0
4,subject8835,Does anyone else find it offensive that I hap...,2012-03-14 18:31:05,reddit post,[removed],0


In [12]:
df_test[["ID", "label"]].head()



,ID,label
0,subject8835,0
1,subject8835,0
2,subject8835,0
3,subject8835,0
4,subject8835,0


In [13]:
df_val[["ID", "label"]].head()

,ID,label
0,subject6620,0
1,subject6620,0
2,subject6620,0
3,subject6620,0
4,subject6620,0


In [14]:
df_test[["ID", "label"]].head()

,ID,label
0,subject8835,0
1,subject8835,0
2,subject8835,0
3,subject8835,0
4,subject8835,0


In [15]:
df_train['label'].value_counts()

label
0    77382
1     7452
Name: count, dtype: int64

In [16]:
df_test['label'].value_counts()

label
0    552890
1     17619
Name: count, dtype: int64

In [17]:
df_val['label'].value_counts()

label
0    151085
1     17422
Name: count, dtype: int64

In [18]:
df_test['ID'].value_counts()

ID
subject5644    2000
subject3275    2000
subject1448    2000
subject8801    2000
subject7475    2000
               ... 
subject6969      10
subject2624      10
subject1158      10
subject7815      10
subject798       10
Name: count, Length: 815, dtype: int64

In [19]:
df_train['ID'].value_counts()

ID
subject2472    1999
subject2419    1999
subject1195    1998
subject3232    1989
subject2129    1983
               ... 
subject2346      22
subject3883      17
subject3640      14
subject8720      12
subject2167       9
Name: count, Length: 152, dtype: int64

In [20]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 570509 entries, 0 to 570508
Data columns (total 6 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   ID      570509 non-null  object
 1   TITLE   570509 non-null  object
 2   DATE    570509 non-null  object
 3   INFO    570509 non-null  object
 4   TEXT    570509 non-null  object
 5   label   570509 non-null  object
dtypes: object(6)
memory usage: 26.1+ MB


In [21]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84834 entries, 0 to 84833
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   ID      84834 non-null  object
 1   TITLE   84834 non-null  object
 2   DATE    84834 non-null  object
 3   INFO    84834 non-null  object
 4   TEXT    84834 non-null  object
 5   label   84834 non-null  object
dtypes: object(6)
memory usage: 3.9+ MB


In [22]:
df_val.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168507 entries, 0 to 168506
Data columns (total 6 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   ID      168507 non-null  object
 1   TITLE   168507 non-null  object
 2   DATE    168507 non-null  object
 3   INFO    168507 non-null  object
 4   TEXT    168507 non-null  object
 5   label   168507 non-null  object
dtypes: object(6)
memory usage: 7.7+ MB


In [23]:
df_train.groupby("label")["ID"].nunique()

label
0    132
1     20
Name: ID, dtype: int64

In [24]:
df_test.groupby("label")["ID"].nunique()


label
0    742
1     73
Name: ID, dtype: int64

In [25]:
df_val.groupby("label")["ID"].nunique()

label
0    279
1     41
Name: ID, dtype: int64

In [26]:
df_test.to_csv("/kaggle/working/df_testERisk19.csv", index=False)
df_train.to_csv("/kaggle/working/df_trainERisk19.csv", index=False)
df_val.to_csv("/kaggle/working/df_valERisk19.csv", index=False)

In [27]:
df_train[df_train["label"] == "1"].groupby("ID")["TEXT"].count()

ID
subject1113      62
subject1637    1263
subject1913      44
subject1953      29
subject3094      63
subject3132     259
subject3259     598
subject3883      17
subject5127     416
subject5711     459
subject6210     542
subject6446      66
subject6681     103
subject7221      30
subject758      129
subject845       75
subject8657    1098
subject8720      12
subject8986     963
subject9229    1224
Name: TEXT, dtype: int64

In [28]:
freq_label1 = (
    df_train[df_train["label"] == "1"]
    .groupby("ID")
    .size()
    .reset_index(name="frequency")
    .sort_values("frequency", ascending=False)
)

In [29]:
freq_label1

,ID,frequency
1,subject1637,1263
19,subject9229,1224
16,subject8657,1098
18,subject8986,963
6,subject3259,598
10,subject6210,542
9,subject5711,459
8,subject5127,416
5,subject3132,259
14,subject758,129


In [30]:
freq_label0 = (
    df_train[df_train["label"] == "0"]
    .groupby("ID")
    .size()
    .reset_index(name="frequency")
    .sort_values("frequency", ascending=False)
)

In [31]:
freq_label0 

,ID,frequency
54,subject2472,1999
50,subject2419,1999
6,subject1195,1998
87,subject3232,1989
41,subject2129,1983
...,...,...
81,subject31,26
75,subject2994,24
47,subject2346,22
115,subject3640,14


###For eRisk24

In [32]:
#zip_path2 = "/kaggle/input/my-erisk24/eRisk 2024 T2"   # replace this according to need
#extract_to2 = "/kaggle/working/my-eRisk24_extracted"        # can be any folder

In [33]:
#extracted2 = extract_zip_recursive(zip_path2, extract_to2)
#unzip_recursive(extract_to)

In [34]:
# 1. Load golden truth labels
truth_dict_combinedeRisk24 = load_golden_truth(f"/kaggle/input/my-erisk24/eRisk 2024 T2")

# 2. Build each split
combinedeRisk24_xml_paths = collect_xml_files(
    f"/kaggle/input/my-erisk24/eRisk 2024 T2"
)
df_combinedeRisk24 = form_dataframe(combinedeRisk24_xml_paths, truth_dict_combinedeRisk24)



In [35]:
df_combinedeRisk24.head()

,ID,TITLE,DATE,INFO,TEXT,label
0,subject902,None,2023-01-24 03:03:38,None,"Sou mesmo. O MS Barney diz pula, eu pergunto q...",0
1,subject902,None,2023-01-24 01:17:15,Reddit post,"Em geral qd vc tem comida, emprego e casa cost...",0
2,subject902,I need a season of Avatar Korra where she beco...,2023-01-19 18:36:41,Reddit post,None,0
3,subject902,None,2023-01-17 02:09:16,Reddit post,Be gay do crime,0
4,subject902,I have no context for all these recent posts,2023-01-17 02:02:39,Reddit post,None,0


In [36]:
df_combinedeRisk24.groupby("label")["ID"].nunique()

label
0    692
1     92
Name: ID, dtype: int64

In [37]:
df_combinedeRisk24.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366886 entries, 0 to 366885
Data columns (total 6 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   ID      366886 non-null  object
 1   TITLE   103694 non-null  object
 2   DATE    366886 non-null  object
 3   INFO    366724 non-null  object
 4   TEXT    283476 non-null  object
 5   label   366886 non-null  object
dtypes: object(6)
memory usage: 16.8+ MB


In [38]:
df_combinedeRisk24['label'].value_counts()

label
0    338843
1     28043
Name: count, dtype: int64

In [39]:
df_combinedeRisk24.to_csv("/kaggle/working/df_combinedeRisk24.csv", index=False)